In [ ]:
## import dependencies
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import pandas as pd
import torch
import plotly
import requests
#import STalign
#from STalign import STalign
import pickle
import json
import cv2
import scanpy as sc
import anndata as ad
import muon
from scipy import sparse

# make plots bigger
plt.rcParams["figure.figsize"] = (12,10)

### MALDI data

In [ ]:
he_id = 'V11L12-109_B1'

In [ ]:
adata = sc.read_h5ad(f'./{he_id}_28_peaks.adata')
adata

In [ ]:
with open(f'./{he_id}_fmp_cellvitplusplus_embed.pickle', 'rb') as handle:
    final_embed_norm_recon_df = pickle.load(handle)

In [ ]:
final_embed_norm_recon_df

### train the model

In [ ]:
import sys
import os

# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("../SuperFocus")

# Add the directory to sys.path
sys.path.append(script_dir)

from importlib import reload
import src.model_training.model_training_orig
import src.model_training.processing_image_features
reload(src.model_training.model_training_orig)
reload(src.model_training.processing_image_features)
from src.model_training.model_training_orig import training_orig

In [ ]:
sample_id = he_id+'_plusplus_sub2_norm'
adata_sub = adata

In [ ]:
from catalyst.utils import set_global_seed
from torch import nn
set_global_seed(123)
training_orig(adata=adata_sub, pt_model_dir=f'../CellViT/maldi_mousebrain/preprocessing/{he_id}/{he_id}/cell_detection', radius=564./2, feature_list=list(adata_sub.var_names), 
              save_model_dir=f'./logs_{sample_id}_fmp', hidden_dims=[1024, 1024, 512, 512], activations=None, residual=False, norm=False, out_act=None, 
              include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite_shrink', kfold=5, cell_embed_dir=f'./{he_id}_fmp_cellvitplusplus_embed.pickle', sort_way='col_row', datatype='dbit')

In [ ]:
import src.post_process
from importlib import reload
reload(src.post_process)
#reload(src.post_process.post_training_processing)
from src.post_process import post_training_processing

In [ ]:
adata_pred = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_fmp/fold_4/checkpoints/model.best.pth', hidden_dims=[1024, 1024, 512, 512], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level1.pickle', feature_list=list(adata_sub.var_names), datatype='protein', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=1, sample_id=sample_id, radius=564./2)

In [ ]:
adata_pred

In [ ]:
adata_pred.var_names

In [ ]:
plt.scatter(adata_pred.obs['x'], adata_pred.obs['y'], c = adata_pred[:, '674.2805'].layers['counts'])

In [ ]:
plt.scatter(adata_pred.obs['x'], adata_pred.obs['y'], c = adata_pred[:, '421.19136'].layers['counts'])

In [ ]:
sc.pl.spatial(adata_pred, img_key=None, color='leiden_res_0.8')

In [ ]:
import pandas as pd

def calculate_9aa_mzs(nt_dict):
    """
    Calculates expected m/z for neurotransmitters in 9-AA negative mode.
    Formula: [M - H]-
    """
    H_mass = 1.007825
    results = []
    
    for name, mass in nt_dict.items():
        mz_negative = mass - H_mass
        results.append({
            "Neurotransmitter": name,
            "Monoisotopic Mass": round(mass, 4),
            "Expected m/z ([M-H]-)": round(mz_negative, 4)
        })
    
    return pd.DataFrame(results)

# Dictionary of common targets and their monoisotopic neutral masses
targets = {
    "GABA": 103.0633,
    "Glutamate": 147.0528,
    "Aspartate": 133.0375,
    "Glutamine": 146.0691,
    "Taurine": 125.0147,
    "N-Acetylaspartate (NAA)": 175.0481,
    "Lactate": 90.0317,
    "Pyruvate": 88.0160,
    "Adenosine": 267.0967
}

# Generate the table
df_mzs = calculate_9aa_mzs(targets)
print(df_mzs)

In [ ]:
import src.model_training.model_training_refined
reload(src.model_training.model_training_refined)
from src.model_training.model_training_refined import training

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata_sub, pt_model_dir=f'../CellViT/maldi_mousebrain/preprocessing/{he_id}/{he_id}/cell_detection', radius=564./2, refine_factor=2, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_fmp/fold_4/checkpoints/model.best.pth',
         save_model_dir=f'./logs_{sample_id}_fmp_2x', hidden_dims=[1024, 1024, 512, 512], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite_shrink', kfold=5, cell_embed_dir=f'./{he_id}_fmp_cellvitplusplus_embed.pickle', 
         lr=1e-7, step_size=25, gamma=0.5, num_epochs=50, sort_way='col_row', random_state=42, datatype='dbit', max_fold=1)

In [ ]:
adata_pred_level2 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_fmp_2x/best_model_fold1.pth', hidden_dims=[1024, 1024, 512, 512], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level2.pickle', feature_list=list(adata_sub.var_names), datatype='protein', regress_out=False, resolutions=[0.4, 0.6],
                                               refine_factor=2, sample_id=sample_id, radius=564./2)

In [ ]:
sc.pl.spatial(adata_pred_level2, img_key=None, color='leiden_res_0.4')

In [ ]:
plt.scatter(adata_pred_level2.obs['x'], adata_pred_level2.obs['y'], c = adata_pred_level2[:, '674.2805'].layers['counts'], s = 4)

In [ ]:
training(adata=adata_sub, pt_model_dir=f'../CellViT/maldi_mousebrain/preprocessing/{he_id}/{he_id}/cell_detection', radius=564./2, refine_factor=4, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_fmp_2x/best_model_fold1.pth',
         save_model_dir=f'./logs_{sample_id}_fmp_4x', hidden_dims=[1024, 1024, 512, 512], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite_shrink', kfold=5, cell_embed_dir=f'./{he_id}_fmp_cellvitplusplus_embed.pickle', 
         lr=1e-7, step_size=25, gamma=0.5, num_epochs=50, sort_way='col_row', random_state=42, datatype='dbit', max_fold=1)

In [ ]:
adata_pred_level4 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_fmp_4x/best_model_fold1.pth', hidden_dims=[1024, 1024, 512, 512], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level4.pickle', feature_list=list(adata_sub.var_names), datatype='protein', regress_out=False, resolutions=[0.4, 0.6],
                                               refine_factor=4, sample_id=sample_id, radius=564./2)

In [ ]:
sc.pl.spatial(adata_pred_level4, img_key=None, color='leiden_res_0.4')

In [ ]:
plt.scatter(adata_pred_level4.obs['x'], adata_pred_level4.obs['y'], c = adata_pred_level4[:, '674.2805'].layers['counts'], s = 2)

In [ ]:
plt.scatter(adata_pred_level4.obs['x'], adata_pred_level4.obs['y'], c = adata_pred_level4[:, '703.13'].layers['counts'], s = 2, vmin=0.2, vmax=1.2)

In [ ]:
training(adata=adata_sub, pt_model_dir=f'../CellViT/maldi_mousebrain/preprocessing/{he_id}/{he_id}/cell_detection', radius=564./2, refine_factor=8, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_fmp_4x/best_model_fold1.pth',
         save_model_dir=f'./logs_{sample_id}_fmp_8x', hidden_dims=[1024, 1024, 512, 512], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite_shrink', kfold=5, cell_embed_dir=f'./{he_id}_fmp_cellvitplusplus_embed.pickle', 
         lr=1e-7, step_size=25, gamma=0.5, num_epochs=50, sort_way='col_row', random_state=42, datatype='dbit', max_fold=1)

In [ ]:
adata_pred_level8 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_fmp_8x/best_model_fold1.pth', hidden_dims=[1024, 1024, 512, 512], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level8.pickle', feature_list=list(adata_sub.var_names), datatype='protein', regress_out=False, resolutions=[0.4, 0.6],
                                               refine_factor=8, sample_id=sample_id, radius=564./2)

In [ ]:
plt.scatter(adata_pred_level8.obs['x'], adata_pred_level8.obs['y'], c = adata_pred_level8[:, '674.2805'].layers['counts'], s = 0.5)

### apply to the whole tissue slide

In [ ]:
from src.model_training.processing_image_features import whole_image_all_cells_features
from src.model_training import MLP

### refine factor is 4

In [ ]:
refine_factor = 4
cell_markers, cell_counts, cell_coords = whole_image_all_cells_features(pt_model_dir=f'../../CellViT/Visium/preprocessing/{he_id}/{he_id}/cell_detection', radius=564./2/refine_factor, cell_embed_dir=f'./{he_id}_fmp_cellvitplusplus_embed.pickle')

In [ ]:
cellvit_marker_array = np.hstack((cell_markers, cell_counts))

In [ ]:
with open(f'./{sample_id}_scaler_refine_level{refine_factor}.pickle', 'rb') as file:
    CODEX_HIST_scaler = pickle.load(file)
    
with open(f'./{sample_id}_gene_exp_pd.pickle', 'rb') as file:
    dbit_pd = pickle.load(file)

In [ ]:
codex_cell_marker_scaled = CODEX_HIST_scaler.transform(cellvit_marker_array)

In [ ]:
hidden_dims=[1024, 1024, 512, 512]
activations=None
residual=False
norm=False
out_act=None
model = MLP.Model_lite_shrink(n_inp=3609, n_out=28, hidden_dims=hidden_dims, activations=activations, out_activation=out_act)

state_dict = torch.load(f'./logs_{sample_id}_fmp_4x/best_model_fold1.pth', map_location=torch.device('cpu')) # Load on CPU\
#state_dict = torch.load(f'./logs_{sample_id}_fmp_4x/best_model_fold1.pth', map_location=torch.device('cpu')) # Load on CPU
model.load_state_dict(state_dict)

model.eval()

with torch.no_grad():
    output_full = model(torch.tensor(codex_cell_marker_scaled).to(torch.float32))

In [ ]:
predicted_gene_full= np.array(output_full)# * cell_counts[:, 0][:, None]
#predicted_gene_full = 10**(predicted_gene_full)-1

## Create adata object
adata = ad.AnnData(predicted_gene_full)#ad.AnnData(sparse.csc_matrix(predicted_gene))

#adata.obs_names = list(dbit_pd.index)
adata.var_names = list(dbit_pd.columns)#list(msi_fmp.columns[2:])

adata.obs['x'] = np.array(cell_coords['pxl_col_in_fullres'])
adata.obs['y'] = np.array(cell_coords['pxl_row_in_fullres'])
adata.obs['section_id'] = [he_id]*adata.n_obs

adata.layers['counts'] = predicted_gene_full.copy()

In [ ]:
adata.uns = {}
adata.uns['spatial'] = {}
adata.uns['spatial'][sample_id] = {}
adata.uns['spatial'][sample_id]['images'] = {}
#adata_query.uns['spatial'][sample_id]['images']['lowres'] = imageLN_flip # rotated_imageLN
adata.uns['spatial'][sample_id]['scalefactors'] = {}
adata.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef'] = 16.0  # 78.5
adata.uns['spatial'][sample_id]['scalefactors']['spot_diameter_fullres'] =  20.0 #2.18
adata.uns['spatial'][sample_id]['scalefactors']['fiducial_diameter_fullres'] = 4.24
adata.uns['spatial'][sample_id]['scalefactors']['tissue_lowres_scalef'] = 16.0 # 78.5

adata.obsm['spatial'] = np.array([adata.obs['x'], adata.obs['y']]).T
adata.obs['sample_id'] = [he_id] * adata.n_obs

In [ ]:
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
#sc.pp.highly_variable_genes(adata, n_top_genes=500)

# scale data, clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')
# plot more PCS
sc.pl.pca(adata, components = ['1,2','3,4','5,6','7,8'], ncols=2)

In [ ]:
sc.pl.pca_variance_ratio(adata)

In [ ]:
# Then we calculate a new umap
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=15, use_rep='X_pca', metric='correlation', random_state=0)

sc.tl.umap(adata, min_dist=0.0, random_state=0)

In [ ]:
sc.tl.leiden(adata, resolution = 0.3, key_added = "leiden_0.3_sc", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
sc.tl.leiden(adata, resolution = 0.4, key_added = "leiden_0.4_sc", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
sc.tl.leiden(adata, resolution = 0.5, key_added = "leiden_0.5_sc", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
sc.pl.umap(adata, color='leiden_0.3_sc')
sc.pl.umap(adata, color='leiden_0.4_sc')
sc.pl.umap(adata, color='leiden_0.5_sc')

In [ ]:
sc.pl.spatial(adata, color='leiden_0.3_sc', img_key=None, size=4.0)

In [ ]:
adata

In [ ]:
refine_factor=1
with open(f'./{sample_id}_training_data_level{refine_factor}.pickle', 'rb') as file:
    x_train1, y_train1, train_coordx1, train_coordy1, x_train_scaled1, other_marker_array_sub1 = pickle.load(file)

In [ ]:
# calculate in validation spots
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, random_state=123, shuffle=True)

for fold, (train_idx, val_idx) in enumerate(kf.split(x_train1)):
    if fold==5:
        val_indx=val_idx

In [ ]:
y_train1_df = pd.DataFrame(y_train1 * x_train1[:, -9][:, None])
pred_y_train1_val = adata_pred.layers['counts'][val_indx, :].copy()
np.sum((pred_y_train1_val - (y_train1 * x_train1[:, -9][:, None])[val_indx, :])**2)/28

In [ ]:
loss = np.array([116723.22589784989,120046.0707,123165.6415,136144.8292])
group = ['1x', '2x', '4x', '8x']

plt.rcParams.update({'font.size': 16, 'axes.titlesize': 25, 'axes.labelsize': 18})

# Replace plt.scatter with plt.plot to connect the points
plt.plot(group, loss, marker='o', linestyle='--', markersize=10, linewidth=1)

plt.xlabel('Training resolution')
plt.ylabel('Validation Loss')

# Remove the top and right frame
plt.gca().spines[['top', 'right']].set_visible(False)

plt.show()

In [ ]:
## Dopamine
plt.scatter(adata.obsm['spatial'][:, 0], adata.obsm['spatial'][:, 1], c = adata[:, '674.2805'].X.toarray(), s = 1.0, vmax=6, vmin=0)

In [ ]:
sample_id

In [ ]:
adata.write_h5ad(f'{sample_id}_MALDI_adata.h5ad')